# Cross-Lingual Abstractive Summarization (EN → TR)
**Course:** CENG 467 — Natural Language Understanding and Generation  
**Task:** Given an English article, generate a Turkish summary  
**Dataset:** WikiLingua (English subset, Hugging Face)  

| Baseline | Approach |
|----------|----------|
| Baseline 1 | EN article → mT5 (EN summary) → MarianMT (TR summary) |
| Baseline 2 | EN article → MarianMT (TR text) → mT5 (TR summary) |

---

## 1. Setup

In [1]:
!pip install datasets transformers sentencepiece rouge_score -q

import json
import torch
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, MarianMTModel, MarianTokenizer
from rouge_score import rouge_scorer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

  Preparing metadata (setup.py) ... done
Device: cuda
GPU: Tesla T4


## 2. Dataset — WikiLingua (English)

In [2]:
print('Loading WikiLingua (english)...')
raw_dataset = load_dataset('wiki_lingua', 'english')
print(f'Train size: {len(raw_dataset["train"]):,}')

Loading WikiLingua (english)...


README.md: 0.00B [00:00, ?B/s]

english/train-00000-of-00001.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/57945 [00:00<?, ? examples/s]

Train size: 57,945


In [3]:
# Extract (EN article, EN summary) pairs
en_texts, en_summaries = [], []

for item in raw_dataset['train']:
    docs = item['article']['document']
    sums = item['article']['summary']
    pairs_d = docs if isinstance(docs, list) else [docs]
    pairs_s = sums if isinstance(sums, list) else [sums]
    for d, s in zip(pairs_d, pairs_s):
        if d and s:
            en_texts.append(d.strip())
            en_summaries.append(s.strip())
    if len(en_texts) >= 1000:
        break

# 80/20 train/test split
split = int(len(en_texts) * 0.8)
train_en,  train_ref = en_texts[:split],  en_summaries[:split]
test_en,   test_ref  = en_texts[split:],  en_summaries[split:]

print(f'Total: {len(en_texts)} | Train: {len(train_en)} | Test: {len(test_en)}')
print(f'\nSample EN article : {test_en[0][:200]}...')
print(f'Sample EN summary : {test_ref[0]}')

Total: 1001 | Train: 800 | Test: 201

Sample EN article : Nursing is a lot like a game of chess.  It’s complex and you have to think several moves ahead.  You need to be able to assess a patient critically to determine his medical needs, while taking multipl...
Sample EN summary : Think critically. Communicate effectively. Be detail-oriented. Organize efficiently. Maintain physical stamina. React and think quickly.


## 3. Models
We load two models used across both baselines:
- **MarianMT** (Helsinki-NLP): EN → TR translation
- **mT5** (csebuetnlp): Multilingual abstractive summarization

In [4]:
# MarianMT: EN → TR
TRANSLATE_MODEL = 'Helsinki-NLP/opus-mt-tc-big-en-tr'
print(f'Loading {TRANSLATE_MODEL}...')
tr_tokenizer = MarianTokenizer.from_pretrained(TRANSLATE_MODEL)
tr_model     = MarianMTModel.from_pretrained(TRANSLATE_MODEL).to(device)
tr_model.eval()
print('Done.')

Loading Helsinki-NLP/opus-mt-tc-big-en-tr...


tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/833k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Done.


In [5]:
# mT5: Multilingual summarization
MT5_MODEL = 'csebuetnlp/mT5_multilingual_XLSum'
print(f'Loading {MT5_MODEL}...')
mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL, use_fast=False)
mt5_model     = AutoModelForSeq2SeqLM.from_pretrained(MT5_MODEL).to(device)
mt5_model.eval()
print('Done.')

Loading csebuetnlp/mT5_multilingual_XLSum...


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Done.


In [6]:
# Core functions
WHITESPACE_HANDLER = lambda k: ' '.join(k.split())

def translate(text, max_chars=1000):
    """Translate English text to Turkish using MarianMT."""
    inputs = tr_tokenizer(
        text[:max_chars], return_tensors='pt',
        truncation=True, padding=True, max_length=512
    ).to(device)
    with torch.no_grad():
        out = tr_model.generate(**inputs, max_length=512)
    return tr_tokenizer.decode(out[0], skip_special_tokens=True)

def summarize_mt5_en(text, max_in=512, max_out=128):
    """Summarize text in English using mT5."""
    input_text = '<english> ' + WHITESPACE_HANDLER(text)
    inputs = mt5_tokenizer(
        input_text, return_tensors='pt',
        max_length=max_in, truncation=True, padding='max_length'
    ).to(device)
    with torch.no_grad():
        out = mt5_model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_out, num_beams=4,
            early_stopping=True, no_repeat_ngram_size=2
        )
    return mt5_tokenizer.decode(out[0], skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)

def summarize_mt5_tr(text, max_in=512, max_out=128):
    """Summarize Turkish text using mT5."""
    input_text = '<turkish> ' + WHITESPACE_HANDLER(text)
    inputs = mt5_tokenizer(
        input_text, return_tensors='pt',
        max_length=max_in, truncation=True, padding='max_length'
    ).to(device)
    with torch.no_grad():
        out = mt5_model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_length=max_out, num_beams=4,
            early_stopping=True, no_repeat_ngram_size=2
        )
    return mt5_tokenizer.decode(out[0], skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)

print('Functions defined.')

Functions defined.


## 4. Reference Summaries
Since WikiLingua has no aligned EN-TR pairs, we translate EN reference summaries  
to Turkish using MarianMT to create evaluation references.

In [7]:
print(f'Translating {len(test_ref)} EN reference summaries to TR...')
test_tr_refs = []
for i, s in enumerate(test_ref):
    test_tr_refs.append(translate(s))
    if (i+1) % 20 == 0:
        print(f'  {i+1}/{len(test_ref)} done...')

print('Done.')
print(f'\nEN ref : {test_ref[0]}')
print(f'TR ref : {test_tr_refs[0]}')

Translating 201 EN reference summaries to TR...
  20/201 done...
  40/201 done...
  60/201 done...
  80/201 done...
  100/201 done...
  120/201 done...
  140/201 done...
  160/201 done...
  180/201 done...
  200/201 done...
Done.

EN ref : Think critically. Communicate effectively. Be detail-oriented. Organize efficiently. Maintain physical stamina. React and think quickly.
TR ref : Eleştirel düşünün. Etkili iletişim kurun. Detay odaklı olun. Verimli bir şekilde organize edin. Fiziksel dayanıklılığı koruyun. Tepki verin ve hızlı düşünün.


## 5. Baseline 1 — Summarize then Translate
**EN article → mT5 (EN summary) → MarianMT (TR summary)**

In [8]:
def baseline1(text):
    """EN article → EN summary (mT5) → TR summary (MarianMT)"""
    en_summary = summarize_mt5_en(text)
    tr_summary = translate(en_summary)
    return en_summary, tr_summary

# Quick test
en_sum, tr_sum = baseline1(test_en[0])
print('EN article :', test_en[0][:200])
print('\nEN summary :', en_sum)
print('TR summary :', tr_sum)
print('TR ref     :', test_tr_refs[0])

EN article : Nursing is a lot like a game of chess.  It’s complex and you have to think several moves ahead.  You need to be able to assess a patient critically to determine his medical needs, while taking multipl

EN summary : In our series of letters from African journalists, Dr John Wright looks at how nurses communicate with their patients.
TR summary : Afrikalı gazetecilerden gelen mektup serimizde, Dr. John Wright hemşirelerin hastalarıyla nasıl iletişim kurduğuna bakıyor.
TR ref     : Eleştirel düşünün. Etkili iletişim kurun. Detay odaklı olun. Verimli bir şekilde organize edin. Fiziksel dayanıklılığı koruyun. Tepki verin ve hızlı düşünün.


## 6. Baseline 2 — Translate then Summarize
**EN article → MarianMT (TR text) → mT5 (TR summary)**

In [9]:
def baseline2(text):
    """EN article → TR translation (MarianMT) → TR summary (mT5)"""
    tr_text    = translate(text)
    tr_summary = summarize_mt5_tr(tr_text)
    return tr_text, tr_summary

# Quick test
tr_text, tr_sum = baseline2(test_en[0])
print('TR text    :', tr_text[:200])
print('\nTR summary :', tr_sum)
print('TR ref     :', test_tr_refs[0])

TR text    : Hemşirelik satranç oyununa çok benzer. Karmaşıktır ve birkaç hamleyi düşünmek zorundasınız. Tıbbi ihtiyaçlarını belirlemek için bir hastayı eleştirel olarak değerlendirebilmeniz gerekir, bu sırada bir

TR summary : Koronavirüsle mücadelede kritik düşünme, hemşirelik satranç oyununa çok benzer.
TR ref     : Eleştirel düşünün. Etkili iletişim kurun. Detay odaklı olun. Verimli bir şekilde organize edin. Fiziksel dayanıklılığı koruyun. Tepki verin ve hızlı düşünün.


## 7. Evaluation — ROUGE Scores

In [10]:
def compute_rouge(preds, refs):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    r1, r2, rl = [], [], []
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {
        'ROUGE-1': round(sum(r1)/len(r1)*100, 2),
        'ROUGE-2': round(sum(r2)/len(r2)*100, 2),
        'ROUGE-L': round(sum(rl)/len(rl)*100, 2),
    }

In [11]:
N = 20  # increase to 50+ for final evaluation

b1_preds, b2_preds = [], []

print(f'Evaluating {N} examples...')
for i in range(N):
    _, b1_out = baseline1(test_en[i])
    b1_preds.append(b1_out)

    _, b2_out = baseline2(test_en[i])
    b2_preds.append(b2_out)

    if (i+1) % 5 == 0:
        print(f'  {i+1}/{N} done...')

print('Evaluation complete.')

Evaluating 20 examples...
  5/20 done...
  10/20 done...
  15/20 done...
  20/20 done...
Evaluation complete.


In [12]:
b1_scores = compute_rouge(b1_preds, test_tr_refs[:N])
b2_scores = compute_rouge(b2_preds, test_tr_refs[:N])

df = pd.DataFrame([
    {'Model': 'Baseline 1 — mT5 (EN sum) + MarianMT (TR)', **b1_scores, 'Status': 'Done'},
    {'Model': 'Baseline 2 — MarianMT (TR) + mT5 (TR sum)', **b2_scores, 'Status': 'Done'},
    {'Model': 'Improved System (planned)',
     'ROUGE-1': '-', 'ROUGE-2': '-', 'ROUGE-L': '-', 'Status': 'Pending'},
])
print(df.to_string(index=False))
print(f'\nEval: {N} examples | References: EN summaries translated to TR via MarianMT')

                                    Model ROUGE-1 ROUGE-2 ROUGE-L  Status
Baseline 1 — mT5 (EN sum) + MarianMT (TR)   13.31    3.04   10.01    Done
Baseline 2 — MarianMT (TR) + mT5 (TR sum)   16.58    3.34   13.15    Done
                Improved System (planned)       -       -       - Pending

Eval: 20 examples | References: EN summaries translated to TR via MarianMT


## 8. Error Analysis

In [13]:
for i in range(3):
    s1 = compute_rouge([b1_preds[i]], [test_tr_refs[i]])
    s2 = compute_rouge([b2_preds[i]], [test_tr_refs[i]])
    print(f'\n--- Example {i+1} ---')
    print(f'EN  : {test_en[i][:150]}...')
    print(f'REF : {test_tr_refs[i]}')
    print(f'B1  : {b1_preds[i]}')
    print(f'B2  : {b2_preds[i]}')
    print(f'ROUGE-L → B1: {s1["ROUGE-L"]}  |  B2: {s2["ROUGE-L"]}')


--- Example 1 ---
EN  : Nursing is a lot like a game of chess.  It’s complex and you have to think several moves ahead.  You need to be able to assess a patient critically to...
REF : Eleştirel düşünün. Etkili iletişim kurun. Detay odaklı olun. Verimli bir şekilde organize edin. Fiziksel dayanıklılığı koruyun. Tepki verin ve hızlı düşünün.
B1  : Afrikalı gazetecilerden gelen mektup serimizde, Dr. John Wright hemşirelerin hastalarıyla nasıl iletişim kurduğuna bakıyor.
B2  : Koronavirüsle mücadelede kritik düşünme, hemşirelik satranç oyununa çok benzer.
ROUGE-L → B1: 8.0  |  B2: 4.65

--- Example 2 ---
EN  : The Facebook icon looks like a blue box with a white "f" in it. If you’re not automatically logged in, log in with your Facebook account. You will hav...
REF : Facebook uygulamasını açın. Arkadaşınızın profiline gidin.  düğmesine dokunun. Menüden Dostluk Gör'e dokunun. Aşağı kaydırın ve Tüm Fotoğrafları Gör'e dokunun.
B1  : Facebook, kullanıcıların arkadaşlarıyla giriş yapmasına izi

## 9. Save Results

In [14]:
results = {
    'dataset': 'WikiLingua (english)',
    'n_samples': N,
    'note': 'References are EN summaries translated to TR via MarianMT.',
    'baseline_1': {
        'approach': 'EN article → mT5 (EN summary) → MarianMT (TR summary)',
        'models': [MT5_MODEL, TRANSLATE_MODEL],
        'scores': b1_scores,
        'predictions': b1_preds,
    },
    'baseline_2': {
        'approach': 'EN article → MarianMT (TR text) → mT5 (TR summary)',
        'models': [TRANSLATE_MODEL, MT5_MODEL],
        'scores': b2_scores,
        'predictions': b2_preds,
    },
    'references_tr': test_tr_refs[:N],
    'references_en': test_ref[:N],
}

with open('results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print('Saved: results.json')
print(f'Baseline 1 — ROUGE-1: {b1_scores["ROUGE-1"]}  ROUGE-L: {b1_scores["ROUGE-L"]}')
print(f'Baseline 2 — ROUGE-1: {b2_scores["ROUGE-1"]}  ROUGE-L: {b2_scores["ROUGE-L"]}')

Saved: results.json
Baseline 1 — ROUGE-1: 13.31  ROUGE-L: 10.01
Baseline 2 — ROUGE-1: 16.58  ROUGE-L: 13.15
